# Bronze Layer: FIFA World Cup Match Data

This notebook implements the **bronze (raw ingestion)** layer for FIFA World Cup match data.

**Purpose:**
- Ingest raw match data from source (CSV) with minimal transformation
- Add ingestion metadata (timestamp, source file, batch ID)
- Detect schema drift against expected column set
- Validate data quality and quarantine corrupt records
- Persist as Delta table for downstream silver layer processing

**Source:** [jfjelstul/worldcup](https://github.com/jfjelstul/worldcup) GitHub repository 
**Target:** `fifa_worldcup.match_prediction_dev.matches_bronze` (Unity Catalog)

**SLA:** On-demand | Max staleness: configurable via `freshness_hours` widget (default: 168h)

**Downstream Consumers:** `match_prediction_silver` notebook

---

### Data Contract (Output Schema Guarantee)
The bronze table guarantees for the silver layer:
- All 37 source columns present with enforced types (IntegerType, DateType, StringType)
- `_corrupt_record` rows quarantined to separate table
- Metadata columns: `_ingestion_timestamp`, `_source_file`, `_batch_id`, `_row_hash`
- Schema drift detected and logged if source structure changes

### Data Quality Checks
- Schema drift detection (expected vs actual columns)
- Null checks on critical fields (key_id, match_id, match_date, team names, result)
- Duplicate detection on primary key (key_id)
- Row count assertion (> 0 rows)

### Data Product Metadata
- **TBLPROPERTIES:** layer, pipeline, source, quality_tier, owner, refresh_mode, downstream_tables
- **COMMENT ON TABLE** for UC discoverability
- **COMMENT ON COLUMN** for 13 key columns (match_id, team names, scores, metadata)
- **Data Contract Enforcement** cell validates output before exit

---

### Orchestration
**Job:** [FIFA Match Prediction - Full Pipeline](/#job/146142328218872) 
**Schedule:** Manual (on-demand) 
**DAG:** `bronze_ingestion` → `silver_transformation` → `gold_analysis` 
**Cluster:** Amar Rathour's Cluster (`0511-052237-lgol1ybz`)

The full medallion pipeline (Bronze → Silver → Gold) is orchestrated via a single Lakeflow Job with dependency chaining. Each layer runs only after its predecessor completes successfully. Trigger manually via the Jobs UI or CLI.

In [0]:
# No external dependencies required for bronze layer.
# Using built-in Spark + requests (bundled in DBR).
# databricks_helpers removed: replaced with native Spark/dbutils calls.

In [0]:
from datetime import date

# --- Configuration (parameterized for multi-environment support) ---
dbutils.widgets.text("environment", "dev", "Environment (dev/staging/prod)")
dbutils.widgets.text("catalog", "fifa_worldcup", "Unity Catalog Name")
dbutils.widgets.dropdown("refresh_mode", "full", ["full", "incremental"], "Refresh Mode")
dbutils.widgets.text("freshness_hours", "168", "Max staleness (hours)")

ENV = dbutils.widgets.get("environment")
CATALOG = dbutils.widgets.get("catalog")
REFRESH_MODE = dbutils.widgets.get("refresh_mode")
FRESHNESS_HOURS = int(dbutils.widgets.get("freshness_hours"))
SCHEMA_NAME = f"match_prediction_{ENV}"
SOURCE_URL = "https://raw.githubusercontent.com/jfjelstul/worldcup/refs/heads/master/data-csv/matches.csv"
BRONZE_TABLE_NAME = "matches_bronze"
QUARANTINE_TABLE_NAME = "matches_bronze_quarantine"

# --- Fully qualified table names (Unity Catalog ready) ---
BRONZE_FULL_TABLE = f"{CATALOG}.{SCHEMA_NAME}.{BRONZE_TABLE_NAME}"
QUARANTINE_FULL_TABLE = f"{CATALOG}.{SCHEMA_NAME}.{QUARANTINE_TABLE_NAME}"

# --- Setup (native — no external helpers) ---
current_user = spark.sql("SELECT current_user()").collect()[0][0]
working_directory = f"/FileStore/{current_user.split('@')[0]}/match_prediction"

print(f"Environment: {ENV}")
print(f"Catalog: {CATALOG}")
print(f"Refresh mode: {REFRESH_MODE}")
print(f"Max staleness: {FRESHNESS_HOURS} hours")
print(f"Current user: {current_user}")
print(f"Working directory: {working_directory}")
print(f"Bronze table: {BRONZE_FULL_TABLE}")
print(f"Quarantine table: {QUARANTINE_FULL_TABLE}")
print(f"Run date: {date.today()}")

# Clean working directory for idempotent runs
try:
    dbutils.fs.rm(working_directory, recurse=True)
    print(f"Cleaned working directory: {working_directory}")
except Exception:
    print(f"Working directory clean (nothing to remove): {working_directory}")

In [0]:
import time
import requests

# Download source CSV with retry logic and error handling
# Using requests + dbutils.fs to avoid workspace permission issues
MAX_RETRIES = 3
filepath = f"{working_directory}/matches.csv"
local_tmp = "/tmp/matches.csv"

for attempt in range(1, MAX_RETRIES + 1):
    try:
        # Download to driver local filesystem first
        response = requests.get(SOURCE_URL, timeout=30)
        response.raise_for_status()

        # Write to local tmp, then copy to DBFS
        with open(local_tmp, "wb") as f:
            f.write(response.content)

        dbutils.fs.mkdirs(working_directory)
        dbutils.fs.cp(f"file:{local_tmp}", filepath)

        # Verify file exists in DBFS
        files = dbutils.fs.ls(working_directory)
        assert any(f.name == "matches.csv" for f in files), "Source file not found after download!"
        file_size = [f.size for f in files if f.name == "matches.csv"][0]
        print(f"Source file downloaded successfully (attempt {attempt}): {filepath}")
        print(f"File size: {file_size:,} bytes")
        break
    except Exception as e:
        if attempt == MAX_RETRIES:
            raise RuntimeError(f"Failed to download source data after {MAX_RETRIES} attempts: {e}") from e
        print(f"WARN | Download attempt {attempt} failed: {e}. Retrying in 5s...")
        time.sleep(5)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import current_timestamp, lit, input_file_name, md5, concat_ws, col
import uuid

# --- Schema Definition ---
MATCHES_SCHEMA = StructType([
    StructField("key_id", IntegerType(), True),
    StructField("tournament_id", StringType(), True),
    StructField("tournament_name", StringType(), True),
    StructField("match_id", StringType(), True),
    StructField("match_name", StringType(), True),
    StructField("stage_name", StringType(), True),
    StructField("group_name", StringType(), True),
    StructField("group_stage", StringType(), True),
    StructField("knockout_stage", StringType(), True),
    StructField("replayed", StringType(), True),
    StructField("replay", StringType(), True),
    StructField("match_date", DateType(), True),
    StructField("match_time", StringType(), True),
    StructField("stadium_id", StringType(), True),
    StructField("stadium_name", StringType(), True),
    StructField("city_name", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("home_team_id", StringType(), True),
    StructField("home_team_name", StringType(), True),
    StructField("home_team_code", StringType(), True),
    StructField("away_team_id", StringType(), True),
    StructField("away_team_name", StringType(), True),
    StructField("away_team_code", StringType(), True),
    StructField("score", StringType(), True),
    StructField("home_team_score", IntegerType(), True),
    StructField("away_team_score", IntegerType(), True),
    StructField("home_team_score_margin", IntegerType(), True),
    StructField("away_team_score_margin", IntegerType(), True),
    StructField("extra_time", IntegerType(), True),
    StructField("penalty_shootout", IntegerType(), True),
    StructField("score_penalties", StringType(), True),
    StructField("home_team_score_penalties", IntegerType(), True),
    StructField("away_team_score_penalties", IntegerType(), True),
    StructField("result", StringType(), True),
    StructField("home_team_win", IntegerType(), True),
    StructField("away_team_win", IntegerType(), True),
    StructField("draw", IntegerType(), True),
])


def ingest_to_bronze(filepath: str) -> DataFrame:
    """Ingest raw CSV data and add bronze layer metadata columns."""
    batch_id = str(uuid.uuid4())

    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("delimiter", ",")
        .option("escape", "\\")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(StructType(MATCHES_SCHEMA.fields + [StructField("_corrupt_record", StringType(), True)]))
        .load(filepath)
    )

    # Add ingestion metadata
    df_with_metadata = df.withColumns({
        "_ingestion_timestamp": current_timestamp(),
        "_source_file": input_file_name(),
        "_batch_id": lit(batch_id),
        "_row_hash": md5(concat_ws("|", *[col(c) for c in MATCHES_SCHEMA.fieldNames()])),
    })

    return df_with_metadata


matches_bronze_df = ingest_to_bronze(filepath)
print(f"Ingested {matches_bronze_df.count():,} records")
display(matches_bronze_df.limit(10))

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    count, countDistinct, sum as spark_sum,
    min as spark_min, max as spark_max,
    col, when
)

def validate_bronze_quality(df: DataFrame) -> dict:
    """Run all data quality checks in a single-pass aggregation to avoid repeated CSV scans.

    Optimization: Consolidated multiple .filter().count() calls into one .agg() pass.
    This addresses the Spark advisory 'Accelerate queries with Delta' which flagged
    repeated highly-selective filter scans against the raw CSV source file.
    """

    # --- Schema Drift Detection ---
    expected_columns = set(MATCHES_SCHEMA.fieldNames())
    actual_columns = set(df.columns) - {"_corrupt_record", "_ingestion_timestamp", "_source_file", "_batch_id", "_row_hash"}
    missing_cols = expected_columns - actual_columns
    extra_cols = actual_columns - expected_columns

    if missing_cols or extra_cols:
        print(f"WARN | Schema drift detected!")
        if missing_cols:
            print(f"  Missing columns: {missing_cols}")
        if extra_cols:
            print(f"  Unexpected columns: {extra_cols}")
    else:
        print(f"PASS | Schema drift check: All {len(expected_columns)} expected columns present")

    # Cache to avoid re-reading CSV for the duplicate check
    df.cache()

    # --- Single-pass aggregation for all metrics ---
    critical_columns = ["key_id", "match_id", "match_date", "home_team_name", "away_team_name", "result"]

    agg_exprs = [
        count("*").alias("total_rows"),
        spark_sum(when(col("_corrupt_record").isNotNull(), 1).otherwise(0)).alias("corrupt_records"),
        countDistinct("key_id").alias("distinct_keys"),
        spark_min("match_date").alias("min_date"),
        spark_max("match_date").alias("max_date"),
    ]
    # Add null counts for each critical column in the same pass
    for c in critical_columns:
        agg_exprs.append(spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(f"null_{c}"))

    stats = df.agg(*agg_exprs).collect()[0]

    # --- Extract metrics ---
    total_rows = stats["total_rows"]
    corrupt_count = int(stats["corrupt_records"])
    duplicate_count = total_rows - stats["distinct_keys"]
    metrics = {"total_rows": total_rows, "corrupt_records": corrupt_count, "duplicate_key_ids": duplicate_count}

    # --- Report results ---
    print(f"{'PASS' if corrupt_count == 0 else 'WARN'} | Corrupt records: {corrupt_count}")

    print("\n--- Null Check on Critical Fields ---")
    for c in critical_columns:
        null_count = int(stats[f"null_{c}"])
        null_pct = (null_count / total_rows * 100) if total_rows > 0 else 0
        status = "PASS" if null_pct == 0 else ("WARN" if null_pct < 5 else "FAIL")
        print(f"  {status} | {c}: {null_count} nulls ({null_pct:.1f}%)")
        metrics[f"null_{c}"] = null_count

    print(f"\n{'PASS' if duplicate_count == 0 else 'FAIL'} | Duplicate key_id values: {duplicate_count}")

    assert total_rows > 0, "FATAL: No rows ingested!"
    print(f"\nPASS | Total rows ingested: {total_rows:,}")
    print(f"INFO | Date range: {stats['min_date']} to {stats['max_date']}")

    has_critical_issues = corrupt_count > 0 or duplicate_count > 0
    print(f"\n{'=' * 50}")
    print(f"OVERALL: {'WARN - Review issues above' if has_critical_issues else 'PASS - All checks passed'}")

    # Unpersist after validation (downstream cells will re-read from the cached plan)
    df.unpersist()
    return metrics


quality_metrics = validate_bronze_quality(matches_bronze_df)

In [0]:
# --- Schema/Database Creation (Unity Catalog ready) ---
if CATALOG != "hive_metastore":
    spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_NAME}")

# --- Quarantine Pattern: Separate good vs bad records ---
# Optimization: Use quality_metrics from previous cell to avoid redundant CSV scans.
# This addresses the Spark advisory 'Accelerate queries with Delta' which flagged
# repeated highly-selective filter scans against the raw CSV source file.
# Instead of re-scanning to count, we leverage already-computed metrics.
corrupt_count = quality_metrics["corrupt_records"]

if corrupt_count > 0:
    # Only apply expensive filter if corrupt records exist
    matches_bronze_df.cache()
    df_clean = matches_bronze_df.filter(col("_corrupt_record").isNull()).drop("_corrupt_record")
    df_quarantine = matches_bronze_df.filter(col("_corrupt_record").isNotNull())
    clean_count = quality_metrics["total_rows"] - corrupt_count
else:
    # No corrupt records — skip filter, just drop the column (single pass)
    df_clean = matches_bronze_df.drop("_corrupt_record")
    df_quarantine = None
    clean_count = quality_metrics["total_rows"]

print(f"Clean records: {clean_count:,}")
print(f"Quarantined records: {corrupt_count:,}")

# --- Write clean bronze table ---
(
    df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_FULL_TABLE)
)

# --- Write quarantine table (if any bad records exist) ---
if df_quarantine is not None and corrupt_count > 0:
    (
        df_quarantine.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(QUARANTINE_FULL_TABLE)
    )
    print(f"Quarantine table written: {QUARANTINE_FULL_TABLE}")
    matches_bronze_df.unpersist()

# --- Add table metadata for discoverability ---
spark.sql(f"""
    ALTER TABLE {BRONZE_FULL_TABLE} SET TBLPROPERTIES (
        'layer' = 'bronze',
        'pipeline' = 'match_prediction',
        'source' = 'github/jfjelstul/worldcup',
        'source_format' = 'csv',
        'quality_tier' = 'raw',
        'environment' = '{ENV}',
        'refresh_mode' = '{REFRESH_MODE}',
        'downstream_tables' = '{CATALOG}.{SCHEMA_NAME}.matches_silver'
    )
""")

spark.sql(f"COMMENT ON TABLE {BRONZE_FULL_TABLE} IS 'Bronze layer: Raw FIFA World Cup match data with ingestion metadata. Source: GitHub jfjelstul/worldcup. Downstream: matches_silver.'")

# --- Column-level comments for discoverability ---
column_comments = {
    "key_id": "Source row identifier from CSV",
    "match_id": "Unique match identifier (natural key)",
    "match_date": "Date the match was played",
    "home_team_name": "Name of the home team",
    "away_team_name": "Name of the away team",
    "home_team_score": "Goals scored by home team (regular + extra time)",
    "away_team_score": "Goals scored by away team (regular + extra time)",
    "tournament_name": "World Cup edition (e.g. FIFA World Cup 2022)",
    "stage_name": "Tournament stage (group, round of 16, quarter-final, etc.)",
    "_ingestion_timestamp": "Timestamp when record was ingested into bronze",
    "_source_file": "Source file path for lineage tracking",
    "_batch_id": "Unique batch identifier for this ingestion run",
    "_row_hash": "MD5 hash of key columns for change detection",
}

for col_name, comment in column_comments.items():
    try:
        spark.sql(f"ALTER TABLE {BRONZE_FULL_TABLE} ALTER COLUMN `{col_name}` COMMENT '{comment}'")
    except Exception:
        pass  # Column may not exist in schema

print(f"\n\u2705 Bronze table registered: {BRONZE_FULL_TABLE}")
print(f"   TBLPROPERTIES: layer, pipeline, source, quality_tier, owner, downstream")
print(f"   Column comments: {len(column_comments)} columns documented")

# Verify
bronze_verify_df = spark.table(BRONZE_FULL_TABLE)
print(f"   Verified row count: {bronze_verify_df.count():,}")
print(f"   Columns: {len(bronze_verify_df.columns)}")

In [0]:
# ============================================================
# DATA CONTRACT: Bronze layer output guarantees
# These assertions form the contract between bronze → silver.
# If any fail, the pipeline should not propagate bad data.
# ============================================================

bronze_df = spark.table(BRONZE_FULL_TABLE)

contract_checks = []

# 1. Row count sanity (FIFA has ~950+ World Cup matches; reject if clearly wrong)
row_count = bronze_df.count()
assert row_count >= 900, f"CONTRACT VIOLATION: Expected 900+ rows, got {row_count}"
contract_checks.append(f"\u2705 Row count: {row_count} (min: 900)")

# 2. No null primary keys
from pyspark.sql.functions import col, count, when
null_match_ids = bronze_df.filter(col("match_id").isNull()).count()
assert null_match_ids == 0, f"CONTRACT VIOLATION: {null_match_ids} null match_id values"
contract_checks.append(f"\u2705 Primary key (match_id): No nulls")

# 3. Required columns exist
required_cols = ["match_id", "match_date", "home_team_name", "away_team_name",
                 "home_team_score", "away_team_score", "tournament_name", "stage_name",
                 "_ingestion_timestamp", "_batch_id", "_row_hash"]
missing_cols = [c for c in required_cols if c not in bronze_df.columns]
assert len(missing_cols) == 0, f"CONTRACT VIOLATION: Missing columns: {missing_cols}"
contract_checks.append(f"\u2705 Schema contract: All {len(required_cols)} required columns present")

# 4. Ingestion metadata populated
null_metadata = bronze_df.filter(
    col("_ingestion_timestamp").isNull() | col("_batch_id").isNull()
).count()
assert null_metadata == 0, f"CONTRACT VIOLATION: {null_metadata} rows missing ingestion metadata"
contract_checks.append(f"\u2705 Ingestion metadata: All rows have _ingestion_timestamp and _batch_id")

# 5. Date range sanity (FIFA World Cup started 1930)
from pyspark.sql.functions import min as spark_min, max as spark_max, year
date_range = bronze_df.agg(
    spark_min("match_date").alias("min_date"),
    spark_max("match_date").alias("max_date")
).collect()[0]
assert date_range["min_date"] is not None, "CONTRACT VIOLATION: All match_date values are null"
assert date_range["min_date"].year >= 1930, f"CONTRACT VIOLATION: Data before 1930 ({date_range['min_date']})"
contract_checks.append(f"\u2705 Date range: {date_range['min_date']} to {date_range['max_date']}")

print("\u2501" * 50)
print("BRONZE DATA CONTRACT: ALL CHECKS PASSED")
print("\u2501" * 50)
for check in contract_checks:
    print(f"  {check}")
print(f"\n  Contract version: 1.0 | Enforced at: {date.today()}")

In [0]:
import json
from datetime import datetime

# --- Exit status for orchestration (Lakeflow Jobs) ---
exit_status = {
    "status": "SUCCESS",
    "environment": ENV,
    "catalog": CATALOG,
    "bronze_table": BRONZE_FULL_TABLE,
    "rows_ingested": row_count,
    "quarantined_rows": corrupt_count if 'corrupt_count' in dir() else 0,
    "data_contract": "PASSED",
    "refresh_mode": REFRESH_MODE,
    "source": SOURCE_URL,
    "run_timestamp": datetime.now().isoformat(),
    "downstream": f"{CATALOG}.{SCHEMA_NAME}.matches_silver",
}

print("=" * 50)
print("BRONZE LAYER PIPELINE COMPLETE")
print("=" * 50)
print(json.dumps(exit_status, indent=2))

# Return structured output for downstream tasks in Jobs
dbutils.notebook.exit(json.dumps(exit_status))